In [0]:
-- 步骤1: 创建临时视图读取Bronze数据
-- 获取 ADF 传递的参数
DECLARE tdspath STRING DEFAULT getArgument('p_tdspath', 'gmall/order_info/default');
CREATE OR REPLACE TEMPORARY VIEW bronze_spu_info_raw AS
SELECT 
    *,
    _metadata.file_path as source_file,
    current_timestamp() as load_timestamp
FROM read_files('abfss://bronze@sahyivy.dfs.core.windows.net/'||:p_tdspath||'/*.parquet');

-- 步骤2: 合并新数据到Silver表（UPSERT）
MERGE INTO silver_spu_info AS target
USING bronze_spu_info_raw AS source
ON target.id = source.id
    WHEN MATCHED THEN
        UPDATE SET                
                target.spu_name = source.spu_name,
                target.description = source.description,
                target.category3_id = source.category3_id,
                target.tm_id = source.tm_id,
                target.create_time  = source.create_time ,
                target.operate_time = source.operate_time,
                target.source_file            =source.source_file           ,
                target.load_timestamp         =source.load_timestamp        ,
                target.update_timestamp = current_timestamp()
    WHEN NOT MATCHED THEN
        INSERT (id          ,
                spu_name    ,
                description ,
                category3_id,
                tm_id       ,
                create_time ,
                operate_time,
                source_file           ,
                load_timestamp        ,
                update_timestamp
                )
        VALUES (source.id          ,
                source.spu_name    ,
                source.description ,
                source.category3_id,
                source.tm_id       ,
                source.create_time ,
                source.operate_time,
                source.source_file           , 
                source.load_timestamp        ,
                current_timestamp()
                );